# Exception Handling in Python

## Why This Matters
Real programs fail — files go missing, users type garbage, networks drop. Exception handling lets you anticipate failure and respond gracefully, turning crashes into helpful messages and recoverable states. Programs without it are fragile; programs with it are robust.

## Table of Contents
1. try / except — the basics
2. Catching specific vs broad exceptions
3. Multiple except blocks
4. The `else` clause — runs if no exception
5. The `finally` clause — always runs
6. `raise` — triggering exceptions manually
7. `raise from` — exception chaining
8. Custom exception classes
9. The exception hierarchy
10. Logging errors vs printing them
11. Common Mistakes
12. Exercises
13. Solutions
14. Mini-Project: Safe Calculator


## 1. try / except — The Basics

**Analogy:** A try/except block is like a safety net under a tightrope walker. The walker (your code) tries to cross normally. If they fall (an exception occurs), the net (except block) catches them so the show doesn't completely stop.

**Key rules:**
- Code in `try` runs normally until an exception happens
- When an exception occurs, Python jumps immediately to `except`
- The rest of the `try` block is skipped
- If no exception occurs, the `except` block is skipped


In [ ]:
# Without exception handling — the program crashes
# result = 10 / 0  # ZeroDivisionError: division by zero  <- uncomment to see

# With exception handling — we catch the crash and handle it
try:
    result = 10 / 0  # This will fail
    print("This line is never reached")  # Skipped after exception
except ZeroDivisionError:
    print("Caught it! Can't divide by zero.")

print("Program continues running after the exception is handled.")


In [ ]:
# Accessing the exception object with 'as e'
try:
    number = int("abc")  # Can't convert 'abc' to int
except ValueError as e:
    print(f"Error type:    {type(e).__name__}")
    print(f"Error message: {e}")
    print(f"Full repr:     {e!r}")


## 2. Catching Specific vs Broad Exceptions

You can catch a **specific** exception type (recommended) or the broad base `Exception` class.

**Key rules:**
- Always catch the MOST SPECIFIC exception you can
- Catching `Exception` (or bare `except:`) hides bugs by swallowing unexpected errors
- A bare `except:` even catches `KeyboardInterrupt` (Ctrl+C) — almost never what you want
- Catching specific exceptions makes code self-documenting: it says *"I know this specific thing can go wrong here"*


In [ ]:
# Specific exception: communicates intent clearly
def divide(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        return None  # We expect this; return a safe value

print(divide(10, 2))   # 5.0
print(divide(10, 0))   # None — handled gracefully

# Too broad — dangerous! This hides ALL errors
def dangerous_parse(s):
    try:
        return int(s)
    except Exception:  # Catches EVERYTHING — bad practice
        return -1  # Is -1 a real value or an error? Ambiguous!

print(dangerous_parse("42"))   # 42  (fine)
print(dangerous_parse("abc"))  # -1  (handled)
# But what if there's a typo bug? It would also return -1 silently!


## 3. Multiple except Blocks

You can chain multiple `except` clauses to handle different exception types differently — like a type-based `if/elif/else`.

**Key rules:**
- Python checks `except` clauses top to bottom and runs the FIRST match
- Put more specific exceptions BEFORE more general ones
- You can catch multiple exceptions in one `except` using a tuple: `except (TypeError, ValueError):`


In [ ]:
def safe_index(lst, idx):
    """Get list[idx] safely, handling both type errors and index errors."""
    try:
        return lst[idx]
    except IndexError:
        print(f"  IndexError: index {idx} is out of range for list of length {len(lst)}")
        return None
    except TypeError:
        print(f"  TypeError: index must be an integer, not {type(idx).__name__}")
        return None

items = ["a", "b", "c"]
print(safe_index(items, 1))      # 'b'
print(safe_index(items, 10))     # IndexError message
print(safe_index(items, "two"))  # TypeError message


In [ ]:
# Catching multiple exceptions in one clause with a tuple
def parse_number(s):
    try:
        return float(s)
    except (ValueError, TypeError) as e:
        # Both ValueError (bad string) and TypeError (None input) handled here
        print(f"Cannot parse {s!r}: {e}")
        return 0.0

print(parse_number("3.14"))  # 3.14
print(parse_number("abc"))   # ValueError
print(parse_number(None))    # TypeError


## 4. The `else` Clause — Runs Only if No Exception

The `else` block runs if the `try` block completed **without raising any exception**.

**Why use it?** It separates "code that might fail" from "code that runs after success". Putting success-path code in `try` can accidentally catch exceptions from that code — `else` avoids that.

**Key rule:** The `else` block is only reached if no exception was raised in `try`.


In [ ]:
def read_and_process(filename):
    try:
        f = open(filename, "r")        # This might fail (FileNotFoundError)
    except FileNotFoundError:
        print(f"File '{filename}' not found.")
    else:
        # This code only runs if open() succeeded
        # Note: exceptions here are NOT caught by the except above
        content = f.read()
        f.close()
        print(f"Successfully read {len(content)} characters.")
        return content
    return None

read_and_process("sample.txt")     # Should succeed
read_and_process("missing.txt")    # File not found


## 5. The `finally` Clause — Always Runs

The `finally` block runs **no matter what** — whether an exception occurred or not, whether it was caught or not. Even if there's a `return` statement in `try` or `except`, `finally` still runs.

**When to use it:** Cleanup code that MUST happen — closing files, releasing locks, disconnecting from a database.

**Note:** The `with` statement is often a cleaner alternative to `finally` for resource cleanup.


In [ ]:
def divide_with_cleanup(a, b):
    print("  Starting operation...")
    try:
        result = a / b
        print(f"  Result: {result}")
        return result
    except ZeroDivisionError:
        print("  Cannot divide by zero!")
        return None
    finally:
        # This ALWAYS runs — even when returning from try or except
        print("  Cleanup: operation complete (finally block).")

print("Case 1: Normal division")
divide_with_cleanup(10, 2)
print()

print("Case 2: Division by zero")
divide_with_cleanup(10, 0)


In [ ]:
# Real-world: simulating a database connection
class FakeDB:
    def __init__(self): self.connected = False
    def connect(self): self.connected = True; print("  DB connected")
    def close(self): self.connected = False; print("  DB connection closed")
    def query(self, sql): return f"Result of: {sql}"

db = FakeDB()

try:
    db.connect()
    result = db.query("SELECT * FROM users")
    print(f"  Query result: {result}")
    # Simulate an unexpected error mid-query
    raise RuntimeError("Network dropped!")
except RuntimeError as e:
    print(f"  Error: {e}")
finally:
    db.close()  # Always close the connection, even after errors

print(f"Connection still open? {db.connected}")


## 6. `raise` — Triggering Exceptions Manually

Use `raise` to **signal** that something has gone wrong in your own code — when you detect an invalid state that Python wouldn't automatically catch.

**Key rules:**
- `raise ExceptionType(message)` — raises a new exception
- `raise` alone (inside an `except` block) — re-raises the current exception
- Choose the right exception type: `ValueError` for bad values, `TypeError` for wrong types, etc.


In [ ]:
def set_age(age):
    """Set a person's age with validation."""
    if not isinstance(age, int):
        raise TypeError(f"Age must be an integer, got {type(age).__name__}")
    if age < 0:
        raise ValueError(f"Age cannot be negative: {age}")
    if age > 150:
        raise ValueError(f"Age {age} seems unrealistic (max 150)")
    return age

# Test valid input
print(set_age(25))

# Test invalid inputs
for test_val in ["twenty", -5, 200]:
    try:
        set_age(test_val)
    except (TypeError, ValueError) as e:
        print(f"set_age({test_val!r}) -> {type(e).__name__}: {e}")


In [ ]:
# Re-raising: catch, log, then re-raise so the caller still sees the error
def process_data(data):
    try:
        return int(data) * 2
    except ValueError as e:
        print(f"[LOG] process_data failed with input {data!r}: {e}")
        raise  # Re-raise the original exception unchanged

try:
    process_data("bad")
except ValueError as e:
    print(f"Caller caught: {e}")


## 7. `raise from` — Exception Chaining

When you catch one exception and want to raise a different (usually higher-level) exception, use `raise NewException(...) from original_exception`. This preserves the original error as context.

**Why:** It lets you provide a user-friendly error at a high level while still preserving the technical root cause for debugging.


In [ ]:
class ConfigError(Exception):
    """Raised when configuration is invalid."""
    pass

def load_config(filename):
    try:
        with open(filename) as f:
            import json
            return json.load(f)
    except FileNotFoundError as e:
        # Chain: ConfigError caused BY FileNotFoundError
        raise ConfigError(f"Config file '{filename}' not found") from e
    except Exception as e:
        raise ConfigError(f"Failed to load config: {e}") from e

try:
    load_config("nonexistent_config.json")
except ConfigError as e:
    print(f"ConfigError: {e}")
    print(f"Caused by:   {e.__cause__}")


## 8. Custom Exception Classes

You can define your own exception types by subclassing `Exception`. This lets callers catch YOUR specific exceptions separately from built-in ones.

**Key rules:**
- Inherit from `Exception` (or a more specific built-in like `ValueError`)
- Use descriptive names ending in `Error` or `Exception`
- Add custom attributes to carry extra info about the error
- Group related exceptions under a common base for easy catching


In [ ]:
# Simple custom exception
class InsufficientFundsError(Exception):
    """Raised when a bank account doesn't have enough money."""

    def __init__(self, balance, amount):
        self.balance = balance
        self.amount  = amount
        self.shortfall = amount - balance
        super().__init__(
            f"Cannot withdraw ${amount:.2f}: balance is ${balance:.2f} "
            f"(${self.shortfall:.2f} short)"
        )

class BankAccount:
    def __init__(self, owner, balance=0):
        self.owner   = owner
        self.balance = balance

    def withdraw(self, amount):
        if amount > self.balance:
            raise InsufficientFundsError(self.balance, amount)
        self.balance -= amount
        return amount

account = BankAccount("Alice", 100)

try:
    account.withdraw(150)
except InsufficientFundsError as e:
    print(f"Error: {e}")
    print(f"Shortfall: ${e.shortfall:.2f}")


In [ ]:
# Hierarchy: group related exceptions under a common base
class AppError(Exception):
    """Base class for all application errors."""

class DatabaseError(AppError):
    """Database-related errors."""

class NetworkError(AppError):
    """Network-related errors."""

class AuthError(AppError):
    """Authentication errors."""

# Now callers can catch all app errors OR specific ones
def risky_operation(mode):
    if mode == "db": raise DatabaseError("Connection refused")
    if mode == "net": raise NetworkError("Timeout")
    if mode == "auth": raise AuthError("Invalid token")

for mode in ["db", "net", "auth"]:
    try:
        risky_operation(mode)
    except DatabaseError as e:
        print(f"DB problem: {e}")
    except AppError as e:  # Catches NetworkError and AuthError
        print(f"App problem ({type(e).__name__}): {e}")


## 9. The Exception Hierarchy

All exceptions in Python are part of a tree. Understanding the hierarchy helps you catch at the right level.

```
BaseException
├── SystemExit          (sys.exit())
├── KeyboardInterrupt   (Ctrl+C)
├── GeneratorExit
└── Exception           ← catch from here downward
    ├── ArithmeticError
    │   ├── ZeroDivisionError
    │   └── OverflowError
    ├── LookupError
    │   ├── IndexError
    │   └── KeyError
    ├── ValueError
    ├── TypeError
    ├── OSError
    │   ├── FileNotFoundError
    │   ├── PermissionError
    │   └── IsADirectoryError
    ├── RuntimeError
    └── StopIteration
```


In [ ]:
# Catching a parent class catches all its children too
def test_lookup(collection, key):
    try:
        return collection[key]
    except LookupError as e:  # Catches both IndexError and KeyError!
        print(f"  Lookup failed ({type(e).__name__}): {e}")
        return None

test_lookup([1, 2, 3], 10)             # IndexError (caught by LookupError)
test_lookup({"a": 1}, "z")             # KeyError   (caught by LookupError)
print(test_lookup({"a": 1}, "a"))      # Returns 1  (no exception)


## 10. Logging Errors vs Printing Them

In production code, use the `logging` module instead of `print()` for error messages. It gives you:
- **Severity levels:** DEBUG, INFO, WARNING, ERROR, CRITICAL
- **Timestamps** and source information automatically
- **Configurable output:** console, file, remote server
- The ability to silence debug output in production


In [ ]:
import logging

# Configure logging
logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s %(levelname)-8s %(message)s",
    datefmt="%H:%M:%S",
)

def parse_int(value):
    try:
        result = int(value)
        logging.debug(f"Successfully parsed {value!r} -> {result}")
        return result
    except ValueError as e:
        logging.error(f"Failed to parse {value!r}: {e}")
        return None
    except TypeError as e:
        logging.error(f"Type error parsing {value!r}: {e}", exc_info=True)  # include traceback
        return None

parse_int("42")
parse_int("abc")
parse_int(None)


## 11. Common Mistakes


In [ ]:
# MISTAKE 1: Bare except — swallows everything including Ctrl+C
# Don't do this:
try:
    x = int("abc")
except:  # catches EVERYTHING: KeyboardInterrupt, SystemExit, bugs...
    print("Something went wrong")  # But WHAT went wrong? No clue.

# Do this instead:
try:
    x = int("abc")
except ValueError as e:
    print(f"Invalid number: {e}")  # Specific, informative


In [ ]:
# MISTAKE 2: Silently swallowing exceptions (pass)
def load_settings(path):
    try:
        with open(path) as f:
            return f.read()
    except Exception:
        pass  # BAD: if this fails, we never know why

result = load_settings("nonexistent.json")
print(f"Result: {result}")  # None — but no error? Something clearly went wrong!

# Better: at minimum, log the error
def load_settings_safe(path):
    try:
        with open(path) as f:
            return f.read()
    except FileNotFoundError:
        logging.warning(f"Settings file not found: {path}. Using defaults.")
        return None


In [ ]:
# MISTAKE 3: Putting too much code in try — hard to know what failed
data = {"key": "abc"}

# BAD: which line raised ValueError? Can't tell.
try:
    value = data["key"]
    number = int(value)
    result = number ** 2
    print(result)
except ValueError:
    print("Something failed")

# BETTER: only the risky part is in try
value = data["key"]
try:
    number = int(value)  # This is the risky line
except ValueError:
    print(f"'{value}' is not a valid number")
else:
    result = number ** 2  # Only runs if int() succeeded
    print(result)


## 12. Exercises

### Exercise 1 (Easy): Safe dictionary access
Write a function `safe_get(dct, key, default=None)` that returns `dct[key]` but returns `default` if the key doesn't exist, and raises a `TypeError` with a clear message if `dct` is not a dictionary.


In [ ]:
# Your code here
def safe_get(dct, key, default=None):
    pass

# Tests:
# print(safe_get({'a': 1}, 'a'))         # 1
# print(safe_get({'a': 1}, 'b'))         # None
# print(safe_get({'a': 1}, 'b', 99))     # 99
# safe_get("not a dict", 'key')          # should raise TypeError


### Exercise 2 (Medium): Retry decorator
Write a function `retry(func, times=3)` that calls `func()` up to `times` times. If it raises an exception, it retries. If all attempts fail, it raises the last exception. If it succeeds, it returns the result.


In [ ]:
# Your code here
def retry(func, times=3):
    pass

# Test it:
import random
attempt_count = 0

def flaky_function():
    global attempt_count
    attempt_count += 1
    print(f"  Attempt {attempt_count}")
    if attempt_count < 3:
        raise ConnectionError("Simulated network error")
    return "Success!"

# result = retry(flaky_function, times=5)
# print(result)  # Should print 'Success!' after 3 attempts


### Exercise 3 (Hard): Custom validation framework
Create a `validate_user(data)` function that takes a dict and validates: `name` (non-empty string), `age` (int, 0-120), `email` (string containing '@'). Define a custom `ValidationError` that stores which field failed and why. Collect ALL validation errors and raise a single exception listing them all.


In [ ]:
# Your code here
class ValidationError(Exception):
    pass

def validate_user(data):
    pass

# Tests:
# validate_user({"name": "Alice", "age": 30, "email": "alice@example.com"})  # should pass
# validate_user({"name": "", "age": -5, "email": "notanemail"})  # should raise ValidationError listing all 3 errors


## 13. Solutions


In [ ]:
# SOLUTION 1 - try it yourself first!
def safe_get(dct, key, default=None):
    if not isinstance(dct, dict):
        raise TypeError(f"Expected a dict, got {type(dct).__name__}")
    try:
        return dct[key]
    except KeyError:
        return default

print(safe_get({"a": 1}, "a"))       # 1
print(safe_get({"a": 1}, "b"))       # None
print(safe_get({"a": 1}, "b", 99))   # 99
try:
    safe_get("not a dict", "key")
except TypeError as e:
    print(f"TypeError: {e}")


In [ ]:
# SOLUTION 2 - try it yourself first!
def retry(func, times=3):
    last_exception = None
    for attempt in range(1, times + 1):
        try:
            return func()
        except Exception as e:
            last_exception = e
            print(f"  Attempt {attempt} failed: {e}")
    raise last_exception

attempt_count = 0

def flaky():
    global attempt_count
    attempt_count += 1
    if attempt_count < 3:
        raise ConnectionError("Simulated network error")
    return "Success!"

result = retry(flaky, times=5)
print(result)


In [ ]:
# SOLUTION 3 - try it yourself first!
class ValidationError(Exception):
    def __init__(self, errors):
        self.errors = errors
        msg = "Validation failed:\n" + "\n".join(f"  - {e}" for e in errors)
        super().__init__(msg)

def validate_user(data):
    errors = []
    name = data.get("name", "")
    if not isinstance(name, str) or not name.strip():
        errors.append("name: must be a non-empty string")
    age = data.get("age")
    if not isinstance(age, int) or not (0 <= age <= 120):
        errors.append(f"age: must be an integer between 0 and 120, got {age!r}")
    email = data.get("email", "")
    if not isinstance(email, str) or "@" not in email:
        errors.append(f"email: must contain '@', got {email!r}")
    if errors:
        raise ValidationError(errors)
    return True

try:
    validate_user({"name": "", "age": -5, "email": "notanemail"})
except ValidationError as e:
    print(e)

print(validate_user({"name": "Alice", "age": 30, "email": "alice@example.com"}))


## 14. Mini-Project: Safe Calculator

Build a calculator that handles every kind of user error gracefully — wrong types, division by zero, overflow, and invalid operations — with friendly messages and proper exception hierarchy.


In [ ]:
import math
import logging

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

# ── Custom exceptions ────────────────────────────────────────────────────────
class CalculatorError(Exception):
    """Base exception for calculator errors."""

class InvalidOperationError(CalculatorError):
    """Raised for unsupported operations."""

class DomainError(CalculatorError):
    """Raised when a mathematical domain constraint is violated."""

# ── Calculator implementation ────────────────────────────────────────────────
class SafeCalculator:
    OPERATIONS = {"+", "-", "*", "/", "**", "sqrt", "log"}

    def calculate(self, op, a, b=None):
        """Perform an operation with full error handling."""
        # Validate inputs
        if op not in self.OPERATIONS:
            raise InvalidOperationError(
                f"Unknown operation '{op}'. Supported: {sorted(self.OPERATIONS)}"
            )
        try:
            a = float(a)
            if b is not None:
                b = float(b)
        except (TypeError, ValueError) as e:
            raise CalculatorError(f"Invalid number input: {e}") from e

        # Perform operation
        try:
            if op == "+":    return a + b
            elif op == "-": return a - b
            elif op == "*": return a * b
            elif op == "/":
                if b == 0:
                    raise DomainError("Cannot divide by zero")
                return a / b
            elif op == "**":
                result = a ** b
                if math.isinf(result):
                    raise OverflowError(f"{a}**{b} is too large to represent")
                return result
            elif op == "sqrt":
                if a < 0:
                    raise DomainError(f"Cannot take square root of negative number: {a}")
                return math.sqrt(a)
            elif op == "log":
                if a <= 0:
                    raise DomainError(f"Logarithm requires positive input: {a}")
                return math.log(a)
        except OverflowError as e:
            raise CalculatorError(f"Overflow: result too large: {e}") from e

calc = SafeCalculator()
print("Calculator created.")


In [ ]:
# Test all cases
test_cases = [
    ("+",    10,    5,    None),
    ("/",    10,    2,    None),
    ("/",    10,    0,    "DomainError: divide by zero"),
    ("sqrt", 16,    None, None),
    ("sqrt", -4,    None, "DomainError: negative sqrt"),
    ("log",  100,   None, None),
    ("log",  0,     None, "DomainError: log of 0"),
    ("**",   2,     1000, "Overflow"),
    ("%",    5,     2,    "Invalid operation"),
    ("+",   "abc",  2,    "Invalid input"),
]

for test in test_cases:
    op, a, b, expected_error = test
    b_display = b if b is not None else ""
    try:
        result = calc.calculate(op, a, b)
        print(f"  {a} {op} {b_display} = {result:.4f}")
    except CalculatorError as e:
        print(f"  {a} {op} {b_display} -> {type(e).__name__}: {e}")
